# Select sample size for PM<sub>2.5</sub> TMREL, RR and BMR

Trying sample size 1000 here

In [1]:
import os
import glob
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === CHOOSE NUMBER OF SAMPLES ===
n_samples = 1000

# Set the seed to produce reproducible random numbers
np.random.seed(42)

In [3]:
# === Set GBD version ===
GBD_version = "GBD23"

In [4]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [8]:
# === Calculate the TMREL distribution ===

# TMREL from GBD23 (uniform distribution) - this hasn't changed since GBD15
tmrel_low = 2.4
tmrel_high = 5.9
tmrel_samples = np.random.uniform(tmrel_low, tmrel_high, size=n_samples)

tmrel_da = xr.DataArray(
    tmrel_samples,
    dims=["samples"],
    coords={"samples": np.arange(n_samples)}
).astype("float32")

# === Save file in scratch directory ===
SAVE_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "TMREL")

out_file = f"{GBD_version}_TMREL_{n_samples}_samples_pm25.nc"
out_path = os.path.join(SAVE_DIR, out_file)
tmrel_da.to_netcdf(out_path)

In [9]:
# === Calculate the RR distribution ===
# Scaled to the TMREL so that RR=1 below the TMREL

RR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / GBD_version / "RR_curves")

for health_VAR in health_vars:
    print(f"Processing {health_VAR}")
    pattern = os.path.join(RR_DIR, f"IHME_GBD_20{GBD_version[-2:]}_AIR_POLLUTION_*_PM_RR_{health_VAR}.nc")
    matches = glob.glob(pattern)
    if len(matches) == 0:
        raise FileNotFoundError(f"No .nc file found for variable: {health_VAR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple .nc files matched for variable {health_VAR}: {matches}")
    RR = xr.open_dataset(matches[0])

    # Updated GBD23 risk curves are log(RR), non updated curves are RR
    if RR["mean"][0] == 1:
        print("Data starts at 1 so they are 'Relative Risk'")
        # Calculate log(RR)
        logRR = np.log(RR)
    elif RR["mean"][0] == 0:
        print("Data starts at 0 so they are 'log(Relative Risk)'")
        # Data is already in logRR format
        logRR = RR
    else:
        raise ValueError(f"Data has unknown start value: {RR['mean'][0]}")

    # Calculate the normal distribution for the logRR
    logRR_mean = logRR["mean"]
    logRR_lower = logRR["lower"]
    logRR_upper = logRR["upper"]

    logRR_std = (logRR_upper - logRR_lower) / (2 * 1.96)

    logRR_samples = xr.DataArray(
        np.random.normal(
            logRR_mean.values[..., np.newaxis],
            logRR_std.values[..., np.newaxis],
            size=logRR_mean.shape + (n_samples,)
        ),
        dims=logRR_mean.dims + ("samples",),
        coords={**logRR_mean.coords, "samples": np.arange(n_samples)},
    )

    # Find the log(RR) at the TMREL
    logRR_tmrel = logRR_samples.sel(exposure=tmrel_da, method="nearest")

    # Shift the function by the log(RR)_TMREL so that log(RR)=0 at TMREL
    logRR_shifted = logRR_samples - logRR_tmrel

    # Set log(RR) below TMREL as 0 and exponentiate to get RR
    RR_samples = np.exp(logRR_shifted.where(logRR_shifted["exposure"] >= tmrel_da, 0))

    # === Save file in scratch directory ===
    SAVE_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "rr_pm25")

    out_file = f"{GBD_version}_RR_{health_VAR}_{n_samples}_samples_pm25.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    RR_samples.to_netcdf(out_path)

print("All processing complete.")

Processing COPD
Data starts at 1 so they are 'Relative Risk'
Saving to /glade/derecho/scratch/awells/EU_pm/rr_pm25/GBD23_RR_COPD_1000_samples_pm25.nc
Processing DIABETES
Data starts at 1 so they are 'Relative Risk'
Saving to /glade/derecho/scratch/awells/EU_pm/rr_pm25/GBD23_RR_DIABETES_1000_samples_pm25.nc
Processing ISCHEMIC_HEART_DISEASE
Data starts at 0 so they are 'log(Relative Risk)'
Saving to /glade/derecho/scratch/awells/EU_pm/rr_pm25/GBD23_RR_ISCHEMIC_HEART_DISEASE_1000_samples_pm25.nc
Processing LOWER_RESPIRATORY_INFECTIONS
Data starts at 0 so they are 'log(Relative Risk)'
Saving to /glade/derecho/scratch/awells/EU_pm/rr_pm25/GBD23_RR_LOWER_RESPIRATORY_INFECTIONS_1000_samples_pm25.nc
Processing LUNG_CANCER
Data starts at 1 so they are 'Relative Risk'
Saving to /glade/derecho/scratch/awells/EU_pm/rr_pm25/GBD23_RR_LUNG_CANCER_1000_samples_pm25.nc
Processing STROKE
Data starts at 0 so they are 'log(Relative Risk)'
Saving to /glade/derecho/scratch/awells/EU_pm/rr_pm25/GBD23_RR_STR

In [12]:
# === Path config ===
EU_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
MASK_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")

# Save file in scratch directory ~25GB
SAVE_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "BMR")

In [16]:
# Load one SHERPA file for lat/lon contraints and resolution
eu_file = "EU_concentration_H_2040.nc"
eu_path = os.path.join(EU_DIR, eu_file)
eu = xr.open_dataarray(eu_path)

# Load country mask
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASK_DIR, mask_file)
mask = xr.open_dataarray(mask_path)

# Crop mask to EU bounding box (with buffer for edge safety)
lat_min, lat_max = float(eu.latitude.min()), float(eu.latitude.max())
lon_min, lon_max = float(eu.longitude.min()), float(eu.longitude.max())
buffer = 0.5

mask_eu = mask.sel(
    lat=slice(lat_min - buffer, lat_max + buffer),
    lon=slice(lon_min - buffer, lon_max + buffer),
)

# Collapse (country, lat, lon) from masks -> single country-index grid
# argmax gives the index of the first country with mask==1 in each cell.
# Cells with no country (all zeros, e.g. ocean) will incorrectly get
# index 0 -- so mask those out explicitly using a coverage count.
country_idx = mask_eu.argmax(dim="country")  # int, shape (lat, lon) at 0.1x0.1
has_country = mask_eu.sum(dim="country") > 0
country_idx = country_idx.where(has_country)  # NaN where no country

# Nearest-neighbor regrid the country-index grid onto EU coords
# This is the only "resampling" step, and it's nearest-neighbor because
# country ID is categorical.
country_idx_eu_res = country_idx.interp(
    lat=eu.latitude, lon=eu.longitude, method="nearest"
)

In [21]:
def build_value_grid_samples(bmr, country_names, country_idx_eu_res, eu):
    """
    bmr: DataArray with dims (country, samples) or (samples, country) — order-agnostic
    country_names: mask.country.values, ordered to match argmax indices in country_idx_eu_res
    country_idx_eu_res: (lat, lon) grid of country indices (float, NaN where no country)
    eu: reference grid for output coords

    Returns: DataArray (latitude, longitude, samples)
    """
    # Force a known order so positional indexing below is safe regardless
    # of how bmr was constructed
    bmr = bmr.transpose("country", "samples")
    bmr_lookup = bmr.values                     # now guaranteed (n_bmr_countries, n_samples)
    bmr_country_list = list(bmr.country.values)
    n_samples = bmr.sizes["samples"]

    idx_to_value = np.full((len(country_names), n_samples), np.nan, dtype=np.float32)
    missing = []
    for i, cname in enumerate(country_names):
        if cname in bmr_country_list:
            idx_to_value[i, :] = bmr_lookup[bmr_country_list.index(cname), :]
        else:
            missing.append(cname)

    if missing:
        print(f"NOTE: {len(missing)} mask countries have no BMR match "
              f"(fine if outside Europe): {missing[:10]}{'...' if len(missing) > 10 else ''}")

    flat_idx = country_idx_eu_res.values
    valid = ~np.isnan(flat_idx)

    out = np.full(flat_idx.shape + (n_samples,), np.nan, dtype=np.float32)
    out[valid] = idx_to_value[flat_idx[valid].astype(int), :]

    return xr.DataArray(
        out,
        coords={"latitude": eu.latitude, "longitude": eu.longitude, "samples": bmr.samples},
        dims=["latitude", "longitude", "samples"],
        name="BMR_COPD_samples",
    )

In [24]:
# === Calculate the BMR distribution ===

for health_VAR in health_vars:
    print(f"Processing {health_VAR}")

    # Load BMR for each country (country, quantile)
    bmr_file = f"{GBD_version}_BMR_Country_{health_VAR}_newlabels_2015-2019.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)  # three quantiles

    # BMR from VizHub (normal distribution)
    bmr_mean = BMR.sel(quantile="mean")
    bmr_lower = BMR.sel(quantile="lower")
    bmr_upper = BMR.sel(quantile="upper")
    bmr_std = (bmr_upper - bmr_lower) / (2 * 1.96)

    bmr_samples = np.random.normal(
        bmr_mean,
        bmr_std,
        size=(n_samples, len(BMR.country)))

    bmr_da = xr.DataArray(
        bmr_samples,
        dims=["samples", "country"],
        coords={"samples": np.arange(n_samples), "country": BMR.country}
    ).astype("float32")

    del bmr_samples

    bmr_eu = build_value_grid_samples(bmr_da, mask.country.values, country_idx_eu_res, eu)

    # Save file ~25GB
    out_file = f"{GBD_version}_BMR_Country_Mask_{health_VAR}_{n_samples}_samples_2015-2019.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    bmr_eu.to_netcdf(out_path)

    del bmr_eu

print("All processing complete.")

Processing COPD
Saving to /glade/derecho/scratch/awells/EU_pm/BMR/GBD23_BMR_Country_Mask_COPD_1000_samples_2015-2019.nc
Processing DIABETES
Saving to /glade/derecho/scratch/awells/EU_pm/BMR/GBD23_BMR_Country_Mask_DIABETES_1000_samples_2015-2019.nc
Processing ISCHEMIC_HEART_DISEASE
Saving to /glade/derecho/scratch/awells/EU_pm/BMR/GBD23_BMR_Country_Mask_ISCHEMIC_HEART_DISEASE_1000_samples_2015-2019.nc
Processing LOWER_RESPIRATORY_INFECTIONS
Saving to /glade/derecho/scratch/awells/EU_pm/BMR/GBD23_BMR_Country_Mask_LOWER_RESPIRATORY_INFECTIONS_1000_samples_2015-2019.nc
Processing LUNG_CANCER
Saving to /glade/derecho/scratch/awells/EU_pm/BMR/GBD23_BMR_Country_Mask_LUNG_CANCER_1000_samples_2015-2019.nc
Processing STROKE
Saving to /glade/derecho/scratch/awells/EU_pm/BMR/GBD23_BMR_Country_Mask_STROKE_1000_samples_2015-2019.nc
Processing DEMENTIA
Saving to /glade/derecho/scratch/awells/EU_pm/BMR/GBD23_BMR_Country_Mask_DEMENTIA_1000_samples_2015-2019.nc
All processing complete.
